In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print (device)

cuda:0


In [2]:
train_path = './data/new_train.tsv'
test_path = './data/new_test.tsv'
df_train = pd.read_csv(train_path, sep = '\t', header=None)
df_test = pd.read_csv(test_path, sep = '\t', header=None)
x_train = list(df_train[0])
y_train = torch.tensor((df_train[1])).to(device)
x_test = list(df_test[0])
y_test = torch.tensor(df_test[1]).to(device)
tokenized_texts_train = [text.lower().split() for text in x_train]
tokenized_texts_test = [text.lower().split() for text in x_test]

df_train.head()
print (x_train[:1])
print (tokenized_texts_train[:1])

max_words = max(len(sentence) for sentence in tokenized_texts_train)
print(max_words)
# print (y_train[:5])
# print (len(x_train))
# print (y_train.shape)

['This quiet , introspective and entertaining independent is worth seeking .']
[['this', 'quiet', ',', 'introspective', 'and', 'entertaining', 'independent', 'is', 'worth', 'seeking', '.']]
52


In [1]:
"""
1. 自己建立词表，训练embedding
（与后面直接使用glove进行比较）
"""
from tokenizers import Tokenizer, trainers, pre_tokenizers
from collections import Counter
from utils import text_to_ids

# 统计词频， 目的是选取词频最高的5000词
word_counts = Counter()
for sentence in tokenized_texts_train:
    word_counts.update(sentence)
print ("所有词的数目：", len(word_counts))

# 选取前10000频率的词构建词表
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.most_common(15000): # 词表大小设为 15000
    if word not in vocab:
        vocab[word] = len(vocab)

# 每句话取前50个单词
input_ids_train = text_to_ids(tokenized_texts_train, vocab, max_len=50)
input_ids_test = text_to_ids(tokenized_texts_test, vocab, max_len=50)
# print(tokenized_texts_train[:1])
print(input_ids_train[:1])
print (len(vocab))


NameError: name 'tokenized_texts_train' is not defined

In [4]:
"""
2. 使用glove做embedding
"""
from utils import build_glove_matrix

glove_path = "glove/glove.2024.wikigiga.50d/wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt"
glove_matrix = build_glove_matrix(vocab, glove_path)
print (glove_matrix.shape)

Loading GloVe...
GloVe loaded.
torch.Size([15002, 50])


In [10]:
"""
2.封装训练与推理过程
"""
from utils import RNNClassifier, evaluate, CNNClassifier
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os
import shutil
from torch.utils.tensorboard import SummaryWriter

vocab_size = len(vocab)
embedding_dim = 50
hidden_dim = 32
n_layers = 1
num_classes = 5
batch_size = 64
epoches = 120

def train(use_glove=True, model_type='cnn', kernel_size=3):
    # 用 tensorboard 记录结果
    if use_glove:
        if model_type=='cnn':
            log_dir = f"runs/use_glove-{model_type}-k={kernel_size}"
        else:
            log_dir = f"runs/use_glove-{model_type}"
    else:
        if model_type=='cnn':
            log_dir = f"runs/no_glove-{model_type}-k={kernel_size}"
        else:
            log_dir = f"runs/no_glove-{model_type}"
    if os.path.exists(log_dir):
        shutil.rmtree(log_dir)  # 删除旧日志，避免曲线混淆
    writer = SummaryWriter(log_dir)
    
    dataset = TensorDataset(input_ids_train, y_train)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    dataset_test = TensorDataset(input_ids_test, y_test)
    dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=True)
    
    if not use_glove and model_type == 'rnn':
        model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, n_layers=n_layers, num_classes=num_classes, use_glove=use_glove).to(device)
    elif use_glove and model_type == 'rnn':
        model = RNNClassifier(vocab_size, embedding_dim, hidden_dim, n_layers=n_layers, num_classes=num_classes, use_glove=use_glove, glove_matrix=glove_matrix).to(device)
    elif not use_glove and model_type == 'cnn':
        model = CNNClassifier(vocab_size, embedding_dim, hidden_dim, kernel_size=kernel_size, num_classes=num_classes, use_glove=use_glove).to(device)
    else:
        model = CNNClassifier(vocab_size, embedding_dim, hidden_dim, kernel_size=kernel_size, num_classes=num_classes, use_glove=use_glove, glove_matrix=glove_matrix).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)

    print(f"start training(use_glove={use_glove})(model_type={model_type})")
    if model_type =='cnn':
        print (f"kernel_size = {kernel_size}")
    for epoch in range(epoches):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            # acc
            with torch.no_grad():
                _, pre = torch.max(outputs.detach(), 1)
                total += batch_y.size(0)
                correct += (pre == batch_y).sum().item()
                total_loss += loss.item()

        writer.add_scalar('Loss/Train', total_loss/len(dataloader), epoch+1)
        writer.add_scalar('Accuracy/Train', 100 * correct / total, epoch+1)

        if (epoch+1)%2 == 0:
            test_loss, test_acc = evaluate(model, dataloader_test, device=device, criterion=criterion)
            acc = 100 * correct / total
            
            writer.add_scalar('Loss/Test', test_loss, epoch+1)
            writer.add_scalar('Accuracy/Test', test_acc, epoch+1)
            # print(f"Epoch [{epoch+1}/{epoches}], train_loss: {total_loss/len(dataloader):.4f}, train_acc: {acc:.2f}%, test_loss: {test_loss:.4f}, test_acc: {test_acc:.2f}")
    writer.close()
    print ("training over")          

In [11]:
"""
打印训练与推理结果，先不使用glove，再比较使用glove的结果
"""
train(use_glove=False, model_type='cnn')
train(use_glove=True, model_type='cnn')

train(use_glove=True, model_type='cnn', kernel_size=1)
train(use_glove=True, model_type='cnn', kernel_size=2)

train(use_glove=True, model_type='cnn', kernel_size=5)

train(use_glove=False, model_type='rnn')
train(use_glove=True, model_type='rnn')

start training(use_glove=False)(model_type=cnn)
kernel_size = 3
training over
start training(use_glove=True)(model_type=cnn)
kernel_size = 3
training over
start training(use_glove=True)(model_type=cnn)
kernel_size = 1
training over
start training(use_glove=True)(model_type=cnn)
kernel_size = 2
training over
start training(use_glove=True)(model_type=cnn)
kernel_size = 5
training over
start training(use_glove=False)(model_type=rnn)
training over
start training(use_glove=True)(model_type=rnn)
training over
